In [ ]:
import os

# 1. Step OUT of the broken folder so we don't delete the ground the kernel is standing on
os.chdir('/kaggle/working')

# 2. Safely delete the broken attempts
!rm -rf /kaggle/working/TRELLIS.2

# 3. Tell Git to intercept the broken GitLab link and route to the GitHub mirror globally
!git config --global url."https://github.com/eigen-mirror/eigen.git".insteadOf "https://gitlab.com/libeigen/eigen.git"

# 4. Clone the repository recursively (the bad URL will auto-swap in the background)
!git clone -b main https://github.com/microsoft/TRELLIS.2.git --recursive

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 1. Initialize the secrets client
user_secrets = UserSecretsClient()

# 2. Ask Kaggle for the secret by its LABEL, not its value
hf_token = user_secrets.get_secret("HF")

# 3. Log in using the retrieved token
login(token=hf_token)
print("Getting Tokens Done")

In [ ]:
import os

# 1. Cast a wider net to intercept both URL variations (with and without .git)
!git config --global url."https://github.com/eigen-mirror/eigen.git".insteadOf "https://gitlab.com/libeigen/eigen.git"
!git config --global url."https://github.com/eigen-mirror/eigen.git".insteadOf "https://gitlab.com/libeigen/eigen"

# 2. Wipe the corrupted temporary extension folders left behind by the frozen script
!rm -rf /tmp/extensions

# 3. Navigate to the folder and run the setup script again
os.chdir('/kaggle/working/TRELLIS.2')
!bash setup.sh --basic --flash-attn --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm

In [ ]:
import fileinput
import os

# Target the exact file throwing the error in your traceback
file_to_patch = '/kaggle/working/TRELLIS.2/trellis2/modules/attention/full_attn.py'

if os.path.exists(file_to_patch):
    with fileinput.FileInput(file_to_patch, inplace=True) as file:
        for line in file:
            # Find the exact failing line and replace it
            if 'out = flash_attn.flash_attn_func(q, k, v)' in line:
                # Preserve the original whitespace/indentation
                indent = line[:len(line) - len(line.lstrip())]
                print(f'{indent}import torch.nn.functional as F')
                print(f'{indent}out = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)).transpose(1, 2)')
            else:
                print(line, end='')
    print("Successfully patched full_attn.py for T4 compatibility!")
else:
    print("File not found! Make sure you cloned the repo first.")

In [ ]:
import os
import sys

# 1. Environment Setup
os.chdir('/kaggle/working/TRELLIS.2')
sys.path.append('/kaggle/working/TRELLIS.2')

# 2. Launch with the environment variable forced directly in the shell command
!ATTN_BACKEND="math" python app.py --share --precision fp16 --max_wait_2d 30

In [ ]:
import torch
import gc

def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print("VRAM Cleared. You can try running the generation again with lower settings.")

clear_vram()